# Feature absorption: class-level adaptation of Chanin et al. 2024

Tests whether TopK exhibits feature absorption (a concept's main latent developing holes because cosine-aligned other latents absorbed its direction) and whether rate-KL avoids it. Mechanistic prediction: absorption saves sparsity under TopK for free, but under rate-KL an absorbing latent's firing rate rises toward the union of the merged concepts' frequencies, which the per-latent KL term punishes -- so TopK should show measurable absorption and rate-KL significantly less.

Metric (see `absorption_metric.py` docstring for details and stated caveats): per class, fit a pixel-space logistic probe direction; find the latent whose firing best predicts the class (max F1); an absorption instance is a class sample where that main latent is silent but decoder-cosine-aligned other latents fire and carry >= 50% of the class-direction mass the main latent typically carries. Reported per method: mean absorption rate, main-latent F1, main-latent miss rate.

**Reading the results:** absorption_rate is the headline (prediction: TopK > RateKL, TunedL1 highest of all -- L1 is Chanin et al.'s primary offender). main_f1 is concept-tracking quality (higher = cleaner). main_miss_rate decomposes: misses WITH aligned compensation = absorption; misses without = plain failure-to-track.

**Before running:** Runtime -> Change runtime type -> GPU.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (go to Runtime > Change runtime type > GPU)')

In [ ]:
!git clone https://github.com/willkn/SAE-Gini.git
%cd SAE-Gini/experiments

In [ ]:
# All three datasets, seeds 0-2. Each (dataset, seed) trains RateKL at two
# lambdas + TopK + TunedL1 (4 models) and fits class probes -- roughly
# comparable per-run cost to the rate-KL replication sweep.
for ds in ['fashion_mnist', 'mnist', 'cifar10']:
    for seed in [0, 1, 2]:
        !python absorption_metric.py --dataset {ds} --seed {seed} --rho 0.09 --lambdas 0.001 0.01

In [ ]:
import json, glob, statistics
from collections import defaultdict

agg = defaultdict(lambda: defaultdict(list))
for path in sorted(glob.glob('results/absorption/*_seed*.json')):
    ds = path.split('/')[-1].rsplit('_seed', 1)[0]
    with open(path) as f:
        for model, r in json.load(f).items():
            agg[ds][model].append(r)

for ds, models in agg.items():
    print(f"\n-- {ds} --")
    print(f"{'Model':24s} {'Absorption':>16s} {'MainF1':>14s} {'MainMiss':>14s}")
    for model, runs in models.items():
        def ms(key):
            vals = [r[key] for r in runs]
            s = statistics.stdev(vals) if len(vals) > 1 else 0.0
            return f"{statistics.mean(vals):.4f}+/-{s:.4f}"
        print(f"{model:24s} {ms('mean_absorption_rate'):>16s} {ms('mean_main_f1'):>14s} {ms('mean_main_miss_rate'):>14s}")

In [ ]:
!zip -r absorption_results.zip results/absorption
from google.colab import files
files.download('absorption_results.zip')